# Import functions

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
%run "DataHelpers.ipynb"

# Run all models against all non-PCA feature selections

In [5]:
for modelName in ModelVariant.__members__:
    print(f'*** - Applying {modelName} to features - Start')

    total = len(FeatureVariant)
    counter = 1
    
    for GENE_FILE_VARIANT in FeatureVariant:
        if GENE_FILE_VARIANT == FeatureVariant.AUTOMATED:
            continue

        print(f"{counter}/{total} - FeatureSet {GENE_FILE_VARIANT} - Start")
        
        FILE_PATH = f"../Data/patient_genes_{GENE_FILE_VARIANT}.csv"
        df = pd.read_csv(FILE_PATH)
        
        ### Dataset split: training and test data, with SMOTE and without SMOTE
        X, y, X_train, X_test, y_train, y_test, test_case_ids = split_data(df, "tnbc", True)

        model_unbalanced = getModel(modelName)
        model_cv_unbalanced = getModel(modelName)
        
        model_smote = getModel(modelName)
        model_cv_smote = getModel(modelName)
    
        # Get weights and weighted model
        weights = getDataSetWeights(y_train)
        model_weighted = getWeightedModel(modelName, weights)
        model_cv_weighted = getWeightedModel(modelName, weights)

        scaler = StandardScaler()
        print(f'Apply Scaling - Start')
        X_trainScaled = scaler.fit_transform(X_train)
        X_testScaled  = scaler.transform(X_test) 
        print(f'Apply Scaling - End')
    
        print("\nApplied Smote")
        X_train_smoteScaled, Y_train_smote = apply_smote_to_train(X_trainScaled, y_train)
       
        # Model without SMOTE, unweighted
        y_pred, y_prob = run_model(model_unbalanced, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, False, modelName)
        print_evaluated_model_accuracy(y_test, y_pred)
    
        # Model with SMOTE
        y_pred_smote, y_prob_smote = run_model(model_smote, X_train_smoteScaled, X_testScaled, Y_train_smote, y_test, test_case_ids, True, False, modelName)
        print_evaluated_model_accuracy(y_test, y_pred_smote)
        
        # Model Weighted
        y_pred_weighted, y_prob_weighted = run_model(model_weighted, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, True, modelName)
        print_evaluated_model_accuracy(y_test, y_pred_weighted)
        
        # Get metrics without SMOTE and unweighted
        metrics = run_cross_validation(model_cv_unbalanced, X_train, y_train, y_test, y_pred, y_prob, False, False, modelName)
        
        # Get metrics with SMOTE
        metrics_smote = run_cross_validation(model_cv_smote, X_train, y_train, y_test, y_pred_smote, y_prob_smote, True, False, modelName)
    
        # Get metrics Weighted
        metrics_weighted = run_cross_validation(model_cv_weighted, X_train, y_train, y_test, y_pred_weighted, y_prob_weighted, False, True, modelName)

        print(f"{counter}/{total} - FeatureSet {GENE_FILE_VARIANT} - End")
        counter += 1
    
    print(f'*** - Applying {modelName} to features - End')

*** - Applying SVM to features - Start
1/7 - FeatureSet researchpapers - Start
X_train.shape=(781, 31)
X_test.shape=(196, 31)
y_train.shape=(781,)
y_test.shape=(196,)
Apply Scaling - Start
Apply Scaling - End

Applied Smote
Accuracy: 0.94
Accuracy: 0.93
Accuracy: 0.95
Model validation for SVC:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.9681528662420382, 'recall': 0.8...   
2     {'accuracy': 0.9166666666666666, 'recall': 0.6...   
3     {'accuracy': 0.9230769230769231, 'recall': 0.6...   
4     {'accuracy': 0.9294871794871795, 'recall': 0.6...   
5     {'accuracy': 0.9487179487179487, 'recall': 0.7...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.9784172661870504, 'r...  
2     {'nTNBC': {'precision': 0.9558823529411765, 'r...  
3     {'nTNBC': {'precision': 0.95, 'recall': 0.9637...  
4

# PCA

In [4]:
for modelName in ModelVariant.__members__:
    print(f'*** - Applying {modelName} to features - Start')
    GENE_FILE_VARIANT = "automated"

    FILE_PATH_TRAIN = f"../Data/patient_genes_automated_train.csv"
    FILE_PATH_TEST = f"../Data/patient_genes_automated_test.csv"
    df_train = pd.read_csv(FILE_PATH_TRAIN)
    df_test = pd.read_csv(FILE_PATH_TEST)
        
    ### Dataset split: training and test data, with SMOTE and without SMOTE
    X_train = df_train.drop(columns=["tnbc"])
    y_train = df_train['tnbc']

    X_test = df_test.drop(columns=["tnbc", "case_id"])
    y_test = df_test['tnbc']
    test_case_ids = df_test["case_id"]

    model_unbalanced = getModel(modelName)
    model_cv_unbalanced = getModel(modelName)
    
    model_smote = getModel(modelName)
    model_cv_smote = getModel(modelName)

    # Get weights and weighted model
    weights = getDataSetWeights(y_train)
    model_weighted = getWeightedModel(modelName, weights)
    model_cv_weighted = getWeightedModel(modelName, weights)
    
    scaler = StandardScaler()
    print(f'Apply Scaling - Start')
    X_trainScaled = scaler.fit_transform(X_train)
    X_testScaled  = scaler.transform(X_test) 
    print(f'Apply Scaling - End')

    print("\nApplied Smote")
    X_train_smoteScaled, Y_train_smote = apply_smote_to_train(X_trainScaled, y_train)

    # Model without SMOTE, unweighted
    y_pred, y_prob = run_model(model_unbalanced, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, False, modelName)
    print_evaluated_model_accuracy(y_test, y_pred)

    # Model with SMOTE
    y_pred_smote, y_prob_smote = run_model(model_smote, X_train_smoteScaled, X_testScaled, Y_train_smote, y_test, test_case_ids, True, False, modelName)
    print_evaluated_model_accuracy(y_test, y_pred_smote)
    
    # Model Weighted
    y_pred_weighted, y_prob_weighted = run_model(model_weighted, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, True, modelName)
    print_evaluated_model_accuracy(y_test, y_pred_weighted)
    
    # Get metrics without SMOTE and unweighted
    metrics = run_cross_validation(model_cv_unbalanced, X_train, y_train, y_test, y_pred, y_prob, False, False, modelName)
    
    # Get metrics with SMOTE
    metrics_smote = run_cross_validation(model_cv_smote, X_train, y_train, y_test, y_pred_smote, y_prob_smote, True, False, modelName)

    # Get metrics Weighted
    metrics_weighted = run_cross_validation(model_cv_weighted, X_train, y_train, y_test, y_pred_weighted, y_prob_weighted, False, True, modelName)
   
    print(f'*** - Applying {modelName} to features - End')

*** - Applying SVM to features - Start
Apply Scaling - Start
Apply Scaling - End

Applied Smote
Accuracy: 0.88
Accuracy: 0.90
Accuracy: 0.88


C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\s

Model validation for SVC:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.8789808917197452, 'recall': 0.0...   
2     {'accuracy': 0.8782051282051282, 'recall': 0.0...   
3     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
4     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
5     {'accuracy': 0.8846153846153846, 'recall': 0.0...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.8789808917197452, 'r...  
2     {'nTNBC': {'precision': 0.8782051282051282, 'r...  
3     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
4     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
5     {'nTNBC': {'precision': 0.8846153846153846, 'r...  


C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\s

Model validation for SVC:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.8789808917197452, 'recall': 0.0...   
2     {'accuracy': 0.8782051282051282, 'recall': 0.0...   
3     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
4     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
5     {'accuracy': 0.8846153846153846, 'recall': 0.0...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.8789808917197452, 'r...  
2     {'nTNBC': {'precision': 0.8782051282051282, 'r...  
3     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
4     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
5     {'nTNBC': {'precision': 0.8846153846153846, 'r...  


...

Model validation for LogisticRegression:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.8789808917197452, 'recall': 0.0...   
2     {'accuracy': 0.8782051282051282, 'recall': 0.0...   
3     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
4     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
5     {'accuracy': 0.8846153846153846, 'recall': 0.0...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.8838709677419355, 'r...  
2     {'nTNBC': {'precision': 0.8782051282051282, 'r...  
3     {'nTNBC': {'precision': 0.8896103896103896, 'r...  
4     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
5     {'nTNBC': {'precision': 0.8846153846153846, 'r...  


C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model validation for LogisticRegression:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.8726114649681529, 'recall': 0.0...   
2     {'accuracy': 0.8782051282051282, 'recall': 0.0...   
3     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
4     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
5     {'accuracy': 0.8846153846153846, 'recall': 0.0...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.8831168831168831, 'r...  
2     {'nTNBC': {'precision': 0.8782051282051282, 'r...  
3     {'nTNBC': {'precision': 0.8896103896103896, 'r...  
4     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
5     {'nTNBC': {'precision': 0.8846153846153846, 'r...  


C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Data\998_Software\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model validation for LogisticRegression:
                                                metrics  \
fold                                                      
1     {'accuracy': 0.8726114649681529, 'recall': 0.0...   
2     {'accuracy': 0.8782051282051282, 'recall': 0.0...   
3     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
4     {'accuracy': 0.8846153846153846, 'recall': 0.0...   
5     {'accuracy': 0.8846153846153846, 'recall': 0.0...   

                                            classReport  
fold                                                     
1     {'nTNBC': {'precision': 0.8831168831168831, 'r...  
2     {'nTNBC': {'precision': 0.8782051282051282, 'r...  
3     {'nTNBC': {'precision': 0.8896103896103896, 'r...  
4     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
5     {'nTNBC': {'precision': 0.8846153846153846, 'r...  
*** - Applying LG to features - End
